# Assignment 1 on Colab

Runtime → Change runtime type → **T4 GPU** (free) or L4/A100 (Pro). Run the cells top to bottom. Everything that must survive a disconnect should be saved to Google Drive: data, checkpoints, results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/a1'
os.makedirs(WORK, exist_ok=True)
print(WORK)

## 1. Code
Clone your repository (or upload the starter kit to Drive once and copy it here).

In [ ]:
%cd /content
# !git clone <your-repo-url> a1        # TODO
# %cd a1
!pip -q install regex einops jaxtyping pytest
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 2. Tests
They run on CPU in a few seconds. Fix failures before training anything.

In [ ]:
!python -m pytest -q tests/

## 3. Data
Download the TinyStories files once to Drive (links on Moodle), tokenise them with your tokenizer, then copy `train.bin`/`val.bin` to the local disk for speed at the start of every session.

In [ ]:
# !wget -q -O $WORK/tinystories_train.txt <url>      # TODO, once
# !python scripts/prepare_data.py --vocab $WORK/tok/vocab.json --merges $WORK/tok/merges.txt --input $WORK/tinystories_train.txt --output $WORK/train.bin
!mkdir -p /content/data && cp $WORK/*.bin /content/data/ 2>/dev/null; ls -la /content/data

## 4. Speed of this GPU (Lecture 2's script)

In [ ]:
import torch, time
n, dt = 4096, torch.float16     # bf16 on L4/A100
a = torch.randn(n, n, device='cuda', dtype=dt); b = torch.randn(n, n, device='cuda', dtype=dt)
for _ in range(3): a @ b
torch.cuda.synchronize(); t = time.perf_counter()
for _ in range(20): c = a @ b
torch.cuda.synchronize(); sec = (time.perf_counter() - t) / 20
print(f'{2 * n**3 / sec / 1e12:.1f} TFLOP/s')

## 5. Train
Checkpoints go to Drive (`--out $WORK/runs/...`), so `--resume` continues after a disconnect. Do the resume test on a 200-step run before the reference run.

In [ ]:
!python scripts/train.py --data /content/data/train.bin --val /content/data/val.bin --out $WORK/runs/smoke --steps 200 --eval_every 100 --ckpt_every 100

In [ ]:
# kill the runtime here (Runtime → Interrupt) and rerun with --resume: the log must continue from the last checkpoint
!python scripts/train.py --data /content/data/train.bin --val /content/data/val.bin --out $WORK/runs/smoke --steps 200 --eval_every 100 --ckpt_every 100 --resume

In [ ]:
# the reference run (Table 1 of the handout); 30-60 min on a T4
!python scripts/train.py --data /content/data/train.bin --val /content/data/val.bin --out $WORK/runs/ref --lr_max 1e-3 --steps 24414 --resume

## 6. results.json
Assemble it from the run summaries and check it before submitting.

In [ ]:
!python grading/validate_results.py $WORK/results.json